# 02 · Scoring, step by step (FSDR V4.0), each target separately

Scoring runs after training (`01_training_step_by_step.ipynb`) on the **same run date**. It takes the target lists, builds the panel, and scores **Target 1** and **Target 2 separately**, each into its own output folder.
It then builds the single file to push.

| Target | Group a (`a_constraint`) | Group b (`b_free`) | Action used |
|---|---|---|---|
| **Target 1** (production) | T1C · code 3, the action's offer families | T1F · code 4, L2 DATA / COMBO | `action_original` |
| **Target 2** (challenger) | T2C · CC-v1, relative ARPU band | T2F · CF-v1, relative band, L2 | `action_agent` |
| **Control** | never contacted | | |

| Step | What it does | Writes |
|---|---|---|
| 1 | check that training is done | — |
| 2 | the target lists (upload, or dummy lists), their checks | `data/scoring/<D>/target1/…`, `target2/…`, `control/` |
| 3 | **P1** panel from the lists, sizes, balance | `data/runs/<D>/score/P1_panel/` |
| 4–5 | **Target 1**: P6 offers, then the final files | `data/runs/<D>/score/P6_offers_target1/`, `data/output/<D>/target1/` |
| 6–7 | **Target 2**: the same | `…/P6_offers_target2/`, `data/output/<D>/target2/` |
| 8 | control file and **push** (both targets PASS) | `data/output/<D>/control/`, `data/output/<D>/push/` |
| 9 | validate the file to push | — |

**Up to 6 offers per line (V4.0).** Slots 1–2 are the 2 best monthly offers and slots 3–4 the 2 best weekly ones. Slot 5 is the 3rd monthly offer and slot 6 the 3rd weekly one (rule EXT-6: same universe, above the anchor, not the same service and price). Favourites and forced bundles only fill a slot 5 / 6 left empty. Set `FSDR_RECO_OFFERS=4` in `.env` for the V3.0 file (4 recommended offers).

Decisions use the documented fallbacks of the LLM agents (rule action, NBO order E-O3 + E-O1) until Phases 5–6 build them, and every row says so (`DECIDED_BY`, `score_source`).

In [ ]:
# --- settings -------------------------------------------------------------------
RUN_DATE = "2026-10-22"         # the same run date as the training notebook
GENERATE_TARGETS = True         # True: dummy lists (stratified draw of the eligible base); False: your uploaded lists
FORCE = True                    # scoring is quick: re-run every step by default
PREVIEW_ROWS = 8
# --- paths and helpers (do not edit) ---------------------------------------------
import json, os, sys
from datetime import date
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:      # works from notebooks/ or the repo root
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from nbx.fsdr import config, csvstd, steps

pd.set_option("display.max_columns", 80, "display.width", 220, "display.max_colwidth", 60)
D = date.fromisoformat(RUN_DATE)
TRAINING, SCORING = config.training_folder(RUN_DATE), config.scoring_folder(RUN_DATE)
RUN, CACHE, OUTPUT = config.FSDR_RUNS_DIR / RUN_DATE, config.FSDR_CACHE_DIR / RUN_DATE, config.output_folder(RUN_DATE)


def show(obj, n=PREVIEW_ROWS, title=None):
    """A table (DataFrame, list of dicts, dict or CSV / Parquet / csv.gz path), first n rows."""
    if isinstance(obj, (str, Path)):
        p = Path(obj)
        title = title or (str(p.relative_to(ROOT)) if p.is_relative_to(ROOT) else str(p))
        if not p.exists() or p.stat().st_size == 0:
            print(f"{title}: empty or not written")
            return
        try:
            obj = (csvstd.read_csv(p) if p.name.endswith(".csv.gz") else pd.read_parquet(p) if p.suffix == ".parquet"
                   else pd.read_csv(p) if p.suffix == ".csv" else json.loads(p.read_text(encoding="utf-8")))
        except pd.errors.EmptyDataError:
            print(f"{title}: no rows")
            return
    if isinstance(obj, dict):
        obj = pd.DataFrame([{"key": k, "value": v if not isinstance(v, (dict, list)) else json.dumps(v, default=str)[:150]}
                            for k, v in obj.items()])
    if isinstance(obj, list):
        obj = pd.DataFrame(obj)
    if title:
        display(Markdown(f"**{title}** · {len(obj):,} rows x {obj.shape[1]} columns"))
    display(obj.head(n))


def tree(folder, depth=3, skip_schemas=True):
    """The files of a folder, as a tree with sizes."""
    folder = Path(folder)
    if not folder.exists():
        print(f"{folder}: does not exist yet")
        return
    print(folder)
    for p in sorted(folder.rglob("*")):
        rel = p.relative_to(folder)
        if len(rel.parts) > depth or (skip_schemas and p.name.endswith(".schema.json")):
            continue
        pad = "    " * (len(rel.parts) - 1)
        print(f"{pad}{p.name}{'/' if p.is_dir() else f'  ({p.stat().st_size / 1e3:,.1f} kB)'}")


def step_status(phase_dir, name):
    meta = steps.read_step(Path(phase_dir) / name)
    print(f"{name}: {meta.get('status', 'not run')}  rows {meta.get('rows', {})}  {meta.get('seconds', '-')} s"
          + (f"  reason: {meta['reason']}" if meta.get("reason") else ""))
    return meta


print(f"run date : {RUN_DATE}")
for name, p in (("training", TRAINING), ("scoring", SCORING), ("runs", RUN), ("cache", CACHE), ("output", OUTPUT)):
    print(f"{name:9s}: {p}  ({'found' if p.exists() else 'not yet'})")
print(f"offers per line: {config.RECO_OFFERS}  (FSDR_RECO_OFFERS)")

from nbx.fsdr import targets
from nbx.fsdr.pipelines import scoring_groups

## Step 1 · Training must be done

P1 reads the T3 segments and the T2 features of the same run date. If a step below is not **done**, run the training notebook first.

In [ ]:
for name in ("T0_load", "T2_features", "T3_segments"):
    meta = step_status(RUN / "train", name)
    assert meta.get("status") == "done", f"{name} is not done: run 01_training_step_by_step.ipynb first"
print("training OK")

## Step 2 · The target lists

You get one list per group, chosen outside the engine. Each list has one row per line and at least `access_method_val`; any other column is kept and carried to the details.

```
data/scoring/<D>/target1/a_constraint/target1_a.csv.gz    T1C
data/scoring/<D>/target1/b_free/target1_b.csv.gz          T1F
data/scoring/<D>/target2/a_constraint/target2_a.csv.gz    T2C
data/scoring/<D>/target2/b_free/target2_b.csv.gz          T2F
data/scoring/<D>/control/control.csv.gz                   control
```

**Checks:**
- **Blocking**:
  - a line in two groups;
  - a missing list;
  - no `access_method_val` column.
- **Dropped**, with the reason in `targets_rejected.csv`:
  - not an 8-digit MSISDN;
  - not in the eligible base;
  - Nursering;
  - in another campaign this week.
- **Warning**: a size more than 2 % off the configured size.

In [ ]:
from nbx.fsdr.contract.cache import CacheSource

if GENERATE_TARGETS:
    gen = scoring_groups.generate_targets(D)
    print(f"dummy lists written (scaled x{gen['scale_factor']} to the base): {gen['sizes']}")
folder = scoring_groups.input_folder(D)
print("lists:", {k: str(targets.group_file(folder, k).relative_to(folder)) if targets.group_file(folder, k) else "MISSING"
                 for k in config.TARGET_FILES})
seg = scoring_groups.base_segments(D)
t, issues = targets.read_targets(folder)
kept, rejected, more = targets.check_targets(t, seg, scoring_groups.other_campaigns(CacheSource(D), D), targets.expected_sizes(folder))
print(f"{len(t):,} lines read, {len(kept):,} kept, {len(rejected):,} dropped")
show(issues + more or [{"level": "-", "message": "no issue"}], title="checks")
show(kept.groupby("GROUP_KEY").size().rename("lines").reset_index(), title="kept per group")

## Step 3 · P1 panel

The panel is the uploaded lists, joined with the T3 segment columns of each line. Without lists, it is a stratified random draw.

**Balance (A-31):** the strata shares should be within 2 points between the groups (only significant gaps count).
**Sizes (A-30):** within ±2 % of the configured sizes (25k per group + 20k control in production).

In [ ]:
p1 = scoring_groups.run_p1_panel(D, force=FORCE)
print(f"P1_panel {p1['status'].upper()} -- mode {p1['mode']}, panel {p1['panel']:,} lines, sizes {p1['sizes']}, "
      f"max balance gap {p1['max_balance_gap_points']} points")
assert p1["status"] == "done", "fix the target lists (see the checks above)"
P1 = RUN / "score" / "P1_panel"
show(P1 / "group_sizes.csv")
bal = pd.read_csv(P1 / "balance.csv")
show(bal.sort_values("max_gap_points", ascending=False), n=10, title="balance: largest gaps between groups")

## Step 4 · Target 1, scoring (P6): T1C (a) and T1F (b)

These rules apply only to the two groups of Target 1:

1. **Action** = `action_original`, the production dictionary.
2. **Candidates** per validity (30 days, then 7 days):
   - T1C: the offer families of the action (B-21);
   - T1F: the L2 DATA / COMBO offers, without VOICE (FR-02 / FR-03).
   - Only offers bought in the base in the last 270 days count (B-20b), at most 25 per validity.
3. **Price passes** around the anchor (the customer's average spend, B-26 / B-27):
   - pass 1: anchor < price < anchor + k;
   - pass 2: the missing type;
   - pass 3: nothing found yet.
   - k = 2 or 3 by action for code 3, k = 3 for code 4.
4. **EXT-6 (V4.0)**: a third offer per validity.
5. **10.1 consistency** (T1C only): a customer with no offer is retried once with his second action.

The **group audit** (A-12 universe, A-12b catalogue, A-17 rules version, A-21 no control line, A-30 / A-31) must PASS, or the push is blocked.

In [ ]:
TARGET = "target1"
p6 = scoring_groups.run_p6_offers(D, force=FORCE, target=TARGET)
P6 = RUN / "score" / f"P6_offers_{TARGET}"
print(f"{p6['stage']} DONE{' (skipped)' if p6.get('skipped') else ''} -- group audit {p6['audit_status']}"
      f"{' (PUSH BLOCKED)' if p6['push_blocked'] else ''}, up to {p6['reco_offers']} offers per line")
show(pd.read_csv(P6 / "group_report.csv"), title="per group")
audit = json.loads((P6 / "audit_groups.json").read_text(encoding="utf-8"))
show([{k: c[k] for k in ("id", "rule", "status", "count", "what")} for c in audit["checks"]], n=20, title="group audit")

In [ ]:
monthly = csvstd.read_csv(P6 / f"{config.GROUPS['t1c']['recos_monthly']}{csvstd.SUFFIX}")
display(Markdown("**How the offers were chosen** (`selected_by`: B-28/B-29/B-30 code 3 passes, FR-09/10/11 code 4, "
                 "CC-04 / CF-04 challenger band, EXT-6 = the third offer of V4.0)"))
sel = pd.concat([csvstd.read_csv(P6 / f"{config.GROUPS[k][f'recos_{v}']}{csvstd.SUFFIX}").assign(group=k, validity=v)
                 for k in config.TARGETS[TARGET] for v in ("monthly", "weekly")])
display(sel.pivot_table(index="selected_by", columns=["group", "validity"], values="access_method_val",
                        aggfunc="count", fill_value=0))
per_line = sel.groupby(["group", "validity", "access_method_val"]).size().rename("offers").reset_index()
display(per_line.groupby(["group", "validity"])["offers"].value_counts().unstack(fill_value=0))
show(monthly, title="recos_monthly of t1c")
show(P6 / "traces.csv", n=6)

## Step 5 · Target 1, final files (code 5) → `data/output/<D>/target1/`

Code 5 turns the recommended offers into the campaign file `input_flytxt`, which has 38 columns:

1. Slots 1–4 come from the price rank of the monthly and weekly offers (F-02 / F-03), and slots 5–6 from the 3rd monthly and the 3rd weekly offer.
2. A line with no recommended offer is dropped (F-04).
3. Favourites (F-06) and forced bundles (F-08) only fill a slot 5 / 6 left empty.
4. Slots are cleared for:
   - the same type and price (F-09);
   - the same offer (F-11);
   - conflicting offers (F-12).
5. Scripts in English and Arabic, segment, BTL-taker flags, dates (F-13 … F-19).

The **output checks** A-20 … A-24 run on the file: 38 columns in order, one row per MSISDN, no control line, the dates, every offer in the catalogue, and the segment being one of the 7 actions.
`input_flytxt_target1.csv.gz` is written **only when everything passes** (H-08); otherwise you get `BLOCKED.txt` with the reasons.

| File | Content |
|---|---|
| `input_flytxt_target1.csv.gz` | the file of Target 1 (both groups) |
| `a_constraint/input_flytxt_t1c.csv.gz`, `b_free/input_flytxt_t1f.csv.gz` | the same, per group |
| `a_constraint/details_t1c.csv.gz`, `b_free/details_t1f.csv.gz` | every line of the group with every column |
| `details_target1.parquet` | the big file to validate: profile, segment, both actions, features, anchors, offers, slots, reasons, removals |
| `final_rules_report.csv`, `output_checks.json`, `audit_groups.json`, `summary.json` | the checks and counts |

In [ ]:
fin = scoring_groups.run_target_final(D, TARGET, force=FORCE)
OUT_T = OUTPUT / TARGET
print(f"{TARGET} final {'READY' if fin['push_ready'] else 'BLOCKED (see BLOCKED.txt)'} -- input_flytxt_{TARGET} "
      f"{fin['input_flytxt_rows']:,} rows; details {fin['details_rows']:,} lines x {fin['details_columns']} columns")
show(fin["output_checks"], title="output checks A-20 ... A-24")
show(fin["per_group"], title="per group a / b")
tree(OUT_T, depth=2)

In [ ]:
det = pd.read_parquet(OUT_T / f"details_{TARGET}.parquet")
reasons = pd.concat([det[f"REASON_{i}"].fillna("(empty)").value_counts().rename(f"slot {i}") for i in range(1, 7)], axis=1)
display(Markdown("**What fills each slot** (V4.0: 5 = 3rd monthly, 6 = 3rd weekly; a favourite or a forced offer only "
                 "fills a slot 5 / 6 left empty)"))
display(reasons.fillna(0).astype(int))
flytxt = csvstd.read_csv(OUT_T / f"input_flytxt_{TARGET}{csvstd.SUFFIX}") if fin["push_ready"] else None
if flytxt is not None:
    n = flytxt[[f"destination_offer_name_{i}" for i in range(1, 7)]].ne("").sum(axis=1)
    n.value_counts().sort_index().plot.bar(title=f"{TARGET}: offers per line in the file", figsize=(6, 3))
    plt.tight_layout(); plt.show()
one = det[det["IN_INPUT_FLYTXT"]].iloc[0]
cols = ["GROUP_KEY", "SEGMENT", "ACTION_ORIGINAL", "ACTION_AGENT", "ACTION_SENT", "B26_AVG_REV_MONTHLY", "F05_MONTHLY"]
cols += [c for i in range(1, 7) for c in (f"SLOT_{i}", f"REASON_{i}", f"REMOVED_{i}")]
display(Markdown(f"**One line, end to end** (`details_{TARGET}.parquet` has {det.shape[1]} columns per line)"))
display(one[[c for c in cols if c in one.index]].to_frame("value"))

## Step 6 · Target 2, scoring (P6): T2C (a) and T2F (b), the challenger

These rules apply only to the two groups of Target 2:

1. **Action** = `action_agent`, the agent dictionary.
2. **Challenger anchor** (CC-03 / CF-03): the customer's spend, split data / voice. A customer with no spend gets the median of his micro-segment.
3. **Relative band** (CC-02 / CF-02): price between 1.0× and 1.5× the anchor. UPSELL actions need at least +0.5 OMR, and no offer is ever a downgrade.
   - The pick is the best `score × price` (C-04).
   - If nothing fits, the band is widened once, to 2.0×.
   - Then the Target 1 passes apply as a fallback (logged as `fallback …`).
4. **EXT-6**: the third offer per validity.
5. **10.1 consistency**: T2C only.

In [ ]:
TARGET = "target2"
p6 = scoring_groups.run_p6_offers(D, force=FORCE, target=TARGET)
P6 = RUN / "score" / f"P6_offers_{TARGET}"
print(f"{p6['stage']} DONE{' (skipped)' if p6.get('skipped') else ''} -- group audit {p6['audit_status']}"
      f"{' (PUSH BLOCKED)' if p6['push_blocked'] else ''}, up to {p6['reco_offers']} offers per line")
show(pd.read_csv(P6 / "group_report.csv"), title="per group")
audit = json.loads((P6 / "audit_groups.json").read_text(encoding="utf-8"))
show([{k: c[k] for k in ("id", "rule", "status", "count", "what")} for c in audit["checks"]], n=20, title="group audit")

In [ ]:
monthly = csvstd.read_csv(P6 / f"{config.GROUPS['t2c']['recos_monthly']}{csvstd.SUFFIX}")
display(Markdown("**How the offers were chosen** (`selected_by`: B-28/B-29/B-30 code 3 passes, FR-09/10/11 code 4, "
                 "CC-04 / CF-04 challenger band, EXT-6 = the third offer of V4.0)"))
sel = pd.concat([csvstd.read_csv(P6 / f"{config.GROUPS[k][f'recos_{v}']}{csvstd.SUFFIX}").assign(group=k, validity=v)
                 for k in config.TARGETS[TARGET] for v in ("monthly", "weekly")])
display(sel.pivot_table(index="selected_by", columns=["group", "validity"], values="access_method_val",
                        aggfunc="count", fill_value=0))
per_line = sel.groupby(["group", "validity", "access_method_val"]).size().rename("offers").reset_index()
display(per_line.groupby(["group", "validity"])["offers"].value_counts().unstack(fill_value=0))
show(monthly, title="recos_monthly of t2c")
show(P6 / "traces.csv", n=6)

## Step 7 · Target 2, final files → `data/output/<D>/target2/`

Same code 5 and the same checks as Target 1, on the Target 2 lines only. The files are the same, with `t2c` / `t2f` in their names.

In [ ]:
fin = scoring_groups.run_target_final(D, TARGET, force=FORCE)
OUT_T = OUTPUT / TARGET
print(f"{TARGET} final {'READY' if fin['push_ready'] else 'BLOCKED (see BLOCKED.txt)'} -- input_flytxt_{TARGET} "
      f"{fin['input_flytxt_rows']:,} rows; details {fin['details_rows']:,} lines x {fin['details_columns']} columns")
show(fin["output_checks"], title="output checks A-20 ... A-24")
show(fin["per_group"], title="per group a / b")
tree(OUT_T, depth=2)

In [ ]:
det = pd.read_parquet(OUT_T / f"details_{TARGET}.parquet")
reasons = pd.concat([det[f"REASON_{i}"].fillna("(empty)").value_counts().rename(f"slot {i}") for i in range(1, 7)], axis=1)
display(Markdown("**What fills each slot** (V4.0: 5 = 3rd monthly, 6 = 3rd weekly; a favourite or a forced offer only "
                 "fills a slot 5 / 6 left empty)"))
display(reasons.fillna(0).astype(int))
flytxt = csvstd.read_csv(OUT_T / f"input_flytxt_{TARGET}{csvstd.SUFFIX}") if fin["push_ready"] else None
if flytxt is not None:
    n = flytxt[[f"destination_offer_name_{i}" for i in range(1, 7)]].ne("").sum(axis=1)
    n.value_counts().sort_index().plot.bar(title=f"{TARGET}: offers per line in the file", figsize=(6, 3))
    plt.tight_layout(); plt.show()
one = det[det["IN_INPUT_FLYTXT"]].iloc[0]
cols = ["GROUP_KEY", "SEGMENT", "ACTION_ORIGINAL", "ACTION_AGENT", "ACTION_SENT", "B26_AVG_REV_MONTHLY", "F05_MONTHLY"]
cols += [c for i in range(1, 7) for c in (f"SLOT_{i}", f"REASON_{i}", f"REMOVED_{i}")]
display(Markdown(f"**One line, end to end** (`details_{TARGET}.parquet` has {det.shape[1]} columns per line)"))
display(one[[c for c in cols if c in one.index]].to_frame("value"))

## Step 8 · Control file and the push

- `data/output/<D>/control/control_<D>.csv.gz` holds the control lines, which are never contacted (H-01), with their strata. They are the baseline for measuring uplift.
- `data/output/<D>/push/input_flytxt.csv.gz` holds both target files merged. It is written **only if both targets passed** and no MSISDN is in two targets (A-30); otherwise you get `push/BLOCKED.txt`.

A **human approval** (H-08) is still needed before the file goes to the campaign platform.

In [ ]:
ctrl = scoring_groups.write_control(D)
print(f"control: {ctrl['lines']:,} lines -> {ctrl['file']}")
push = scoring_groups.run_push(D)
print(f"push {'READY' if push['push_ready'] else 'BLOCKED'} -- {push['input_flytxt_rows']:,} rows {push['rows_per_target']}"
      + (f" -- reasons: {push['reasons']}" if push["reasons"] else ""))
tree(OUTPUT, depth=3)

## Step 9 · Validate the file to push

These are the last checks before handing the file over:
- the 38 columns in order;
- one row per MSISDN, as 8-digit text;
- no control line in the file;
- every line found in the details with `IN_INPUT_FLYTXT = True`;
- offers per line between 1 and 6.

In [ ]:
f = csvstd.read_csv(OUTPUT / "push" / f"input_flytxt{csvstd.SUFFIX}")
details = pd.concat([pd.read_parquet(OUTPUT / t / f"details_{t}.parquet") for t in config.TARGETS])
control = csvstd.read_csv(OUTPUT / "control" / f"control_{RUN_DATE}{csvstd.SUFFIX}")
n = f[[f"destination_offer_name_{i}" for i in range(1, 7)]].ne("").sum(axis=1)
checks = {
    "38 columns in order": list(f.columns) == config.INPUT_FLYTXT_COLUMNS,
    "one row per msisdn": bool(f["msisdn"].is_unique),
    "8-digit text msisdn": bool(f["msisdn"].astype(str).str.fullmatch(r"\d{8}").all()),
    "no control line": not (set(f["msisdn"]) & set(control["ACCESS_METHOD_VAL"])),
    "every line in the details": set(f["msisdn"]) == set(details.loc[details["IN_INPUT_FLYTXT"], "ACCESS_METHOD_VAL"]),
    "offers per line 1..6 (recommended + favourites / forced)": bool(n.between(1, 6).all()),
}
show([{"check": k, "status": "PASS" if v else "FAIL"} for k, v in checks.items()], title="push file")
print(f"{len(f):,} lines, {n.mean():.2f} offers per line on average; segment (action) mix:")
display(f["segment"].value_counts().to_frame("lines"))
show(f, n=5, title="input_flytxt.csv.gz (first rows)")